In [18]:
# Algorithm 2 for Reed–Solomon codes over F_257 
# - s = 1 (no multiplicity)
# - (1, k-1)-weighted degree bound D = floor( sqrt( 2 (k-1) n ) )
# - Accept if agreements > D  (i.e., t = D + 1)
# References to chapter:
#   - Weighted degree: Def. 12.2.5
#   - Lemma: deg(Q(X,P(X))) <= D (Lemma 12.2.6)
#   - Algorithm 12.2.2 (Interpolation + Root Finding with t > D)

import numpy as np
from typing import List, Tuple, Dict, Iterable

# Field: F_257 
q = 257  

def mod(x: int) -> int:
    return x % q

def inv(x: int) -> int:
    x %= q
    assert x != 0, "division by zero in F_q"
    return pow(x, q - 2, q)

def eval_poly_univariate(coeffs: List[int], x: int) -> int:
    """Evaluate P(X) with coeffs in low->high order at x in F_q."""
    x %= q
    acc = 0
    p = 1
    for c in coeffs:
        acc = (acc + (c % q) * p) % q
        p = (p * x) % q
    return acc

# Monomials of (1, k-1)-weighted degree <= D 
def monomials_weighted_deg(D: int, k: int) -> List[Tuple[int, int]]:
    """
    Return monomials (i,j) for X^i Y^j with i + (k-1) j <= D.
    Ordered by (weighted degree, then j, then i) for determinism.
    """
    mons = []
    w = k - 1
    for j in range(0, D // max(1, w) + 2):
        for i in range(0, D + 1):
            if i + w * j <= D:
                mons.append((i, j))
    mons.sort(key=lambda t: (t[0] + (k - 1) * t[1], t[1], t[0]))
    return mons

#  Linear algebra over F_q (Gaussian elimination) ----------
def rref_mod(A: np.ndarray) -> Tuple[np.ndarray, List[int]]:
    """
    Reduced Row Echelon Form over F_q. Returns (RREF, pivot_columns).
    """
    A = A.copy() % q
    m, n = A.shape
    row = 0
    pivots: List[int] = []
    for col in range(n):
        # find pivot row
        pivot = None
        for r in range(row, m):
            if A[r, col] % q != 0:
                pivot = r
                break
        if pivot is None:
            continue
        if pivot != row:
            A[[row, pivot]] = A[[pivot, row]]
        # scale to 1
        invp = inv(int(A[row, col]))
        A[row, :] = (A[row, :] * invp) % q
        # eliminate other rows
        for r in range(m):
            if r != row and A[r, col] % q != 0:
                factor = int(A[r, col])
                A[r, :] = (A[r, :] - factor * A[row, :]) % q
        pivots.append(col)
        row += 1
        if row == m:
            break
    return A % q, pivots

def nullspace_mod(A: np.ndarray) -> List[np.ndarray]:
    """
    Basis of the (right) nullspace over F_q: { v in F_q^n : A v = 0 }.
    """
    R, pivots = rref_mod(A)
    m, n = R.shape
    pivot_set = set(pivots)
    free = [j for j in range(n) if j not in pivot_set]
    basis = []
    for f in free:
        v = np.zeros((n,), dtype=np.int64)
        v[f] = 1
        r = 0
        for p in range(n):
            if r < m and p in pivot_set and pivots[r] == p:
                v[p] = (-R[r, f]) % q
                r += 1
        basis.append(v % q)
    return basis

#  Interpolation matrix for Q(X,Y) 
def build_interpolation_matrix(xs: List[int], ys: List[int], mons: List[Tuple[int, int]]) -> np.ndarray:
    """
    A[r, c] = (alpha_r)^i * (y_r)^j for monomial X^i Y^j.
    """
    n = len(xs)
    N = len(mons)
    A = np.zeros((n, N), dtype=np.int64)
    if N == 0:
        return A
    max_i = max(i for i, _ in mons)
    max_j = max(j for _, j in mons)
    # Precompute powers for speed
    X = np.zeros((n, max_i + 1), dtype=np.int64)
    Y = np.zeros((n, max_j + 1), dtype=np.int64)
    for r, x in enumerate(xs):
        X[r, 0] = 1
        for d in range(1, max_i + 1):
            X[r, d] = (X[r, d - 1] * x) % q
    for r, y in enumerate(ys):
        Y[r, 0] = 1
        for d in range(1, max_j + 1):
            Y[r, d] = (Y[r, d - 1] * y) % q
    for r in range(n):
        for c, (i, j) in enumerate(mons):
            A[r, c] = (X[r, i] * Y[r, j]) % q
    return A

# Sudan 2 Decoder (RS-only) 
class Sudan2DecoderRS:
    """
    Algorithm 2 (RS, s=1) with (1, k-1)-weighted degree.
    Interpolation: linear algebra over F_q on monomials {X^i Y^j : i+(k-1)j <= D}.
    Root finding: brute-force all P with deg(P) <= k-1.

    Public methods
    --------------
    - degree_params(n, k) -> (D, t)
    - interpolate_Q(xs, ys, k, D=None) -> dict with {'D','t','mons','cvec'}
    - list_decode(xs, ys, k, D=None, verify=True, return_codewords=True, limit=None)
        -> list of results dicts: {'P','agreement','codeword','zeros_QXP'}
    """

    @staticmethod
    def degree_params(n: int, k: int) -> Tuple[int, int]:
        D = int((2 * (k - 1) * n) ** 0.5)
        return D, D + 1  # threshold t = D + 1

    @staticmethod
    def _ensure_D_large_enough(n: int, k: int, D: int) -> int:
        """
        Ensure #monomials > n (Interpolation Step needs variables > constraints).
        Using |M_D| = |{(i,j): i+(k-1)j <= D}|; increase D minimally if needed.
        """
        def count_mons(D):
            w = k - 1
            cnt = 0
            for j in range(0, D // max(1, w) + 2):
                for i in range(0, D + 1):
                    if i + w * j <= D:
                        cnt += 1
            return cnt
        while count_mons(D) <= n:
            D += 1
        return D

    @staticmethod
    def _eval_Q(cvec: np.ndarray, mons: List[Tuple[int, int]], x: int, y: int) -> int:
        acc = 0
        for coeff, (i, j) in zip(cvec, mons):
            if coeff % q:
                acc = (acc + int(coeff) * pow(x, i, q) * pow(y, j, q)) % q
        return acc

    @staticmethod
    def _enumerate_polys(k: int, limit: int = None) -> Iterable[List[int]]:
        """
        Generate all coeff vectors P[0..k-1] in F_q^k (deg <= k-1).
        Optional 'limit' to stop after some candidates.
        """
        produced = 0
        cur = [0] * k
        def rec(i: int):
            nonlocal produced
            if limit is not None and produced >= limit:
                return
            if i == k:
                produced += 1
                yield cur.copy()
                return
            for c in range(q):
                cur[i] = c
                yield from rec(i + 1)
        yield from rec(0)

    @classmethod
    def interpolate_Q(cls, xs: List[int], ys: List[int], k: int, D: int = None) -> Dict:
        """
        Interpolation step: find nonzero Q(X,Y) s.t. Q(alpha_i, y_i) = 0 for all i,
        with (1, k-1)-weighted deg(Q) <= D. Returns one solution in coefficient vector form.
        """
        assert len(xs) == len(ys), "xs, ys length mismatch"
        n = len(xs)
        if D is None:
            D, t = cls.degree_params(n, k)
        else:
            t = D + 1
        D = cls._ensure_D_large_enough(n, k, D)
        mons = monomials_weighted_deg(D, k)
        A = build_interpolation_matrix([mod(x) for x in xs], [mod(y) for y in ys], mons)
        ns_basis = nullspace_mod(A)
        assert ns_basis, "No nontrivial Q found; increase D."
        cvec = ns_basis[0]  # pick any non-zero solution
        return {"D": D, "t": t, "mons": mons, "cvec": cvec}

    @classmethod
    def list_decode(
        cls,
        xs: List[int],
        ys: List[int],
        k: int,
        D: int = None,
        verify: bool = True,
        return_codewords: bool = True,
        limit: int = None,
    ) -> List[Dict]:
        """
        Full Algorithm 2:
        1) Interpolate Q with (1,k-1)-deg <= D and Q(alpha_i, y_i)=0 for all i.
        2) Brute-force all P with deg(P) <= k-1; keep those with agreements >= t=D+1.
        3) (Optional) Verify #zeros of R(X)=Q(X,P(X)) across the n positions is > D
           (by Lemma deg R <= D, this forces R≡0 as a polynomial).

        Returns sorted list of dicts with P, agreement, (optional) codeword, and zeros_QXP.
        """
        xs = [mod(x) for x in xs]
        ys = [mod(y) for y in ys]
        n = len(xs)

        data = cls.interpolate_Q(xs, ys, k, D)
        D, t, mons, cvec = data["D"], data["t"], data["mons"], data["cvec"]

        out: List[Dict] = []
        for P in cls._enumerate_polys(k, limit=limit):
            # agreements with received word
            agree = sum(int(eval_poly_univariate(P, x) == y) for x, y in zip(xs, ys))
            if agree >= (D + 1):  # t = D+1
                zeros = None
                if verify:
                    zeros = sum(
                        int(cls._eval_Q(cvec, mons, x, eval_poly_univariate(P, x)) == 0)
                        for x in xs
                    )
                    if zeros <= D:
                        continue  # reject if we didn't see > D zeros (safety)
                item = {"P": P, "agreement": agree, "zeros_QXP": zeros}
                if return_codewords:
                    codeword = [eval_poly_univariate(P, x) for x in xs]
                    item["codeword"] = codeword
                out.append(item)

        out.sort(key=lambda d: (-d["agreement"], d["P"]))
        return out

# Interpolation via Linear Algebra (how we build $Q(X,Y)$)

**What the chapter asks for.**  
Algorithm 12.2.2 (with multiplicity $s=1$) needs a non-zero bivariate polynomial
$$
Q(X,Y)=\sum_{(i,j)\in\mathcal M_D} c_{i,j}\,X^i Y^j
$$
of **$(1,k-1)$-weighted degree** at most $D$ (Def. 12.2.5), such that
$$
Q(\alpha_r,\,y_r)=0 \quad \text{for every received point }(\alpha_r,y_r),\ r=1,\dots,n.
$$
Here $\mathcal M_D=\{(i,j):\, i+(k-1)j\le D\}$. We pick $D=\lfloor\sqrt{2(k-1)n}\rfloor$ and the acceptance threshold $t=D+1$.

---

## Turning interpolation into a linear system

Index the monomials $\mathcal M_D=\{(i_\ell,j_\ell)\}_{\ell=1}^{N}$ in a fixed order.  
Each condition $Q(\alpha_r,y_r)=0$ is linear in the unknowns $c_{i_\ell,j_\ell}$:
$$
\sum_{\ell=1}^{N} c_{i_\ell,j_\ell}\,\alpha_r^{\,i_\ell}\,y_r^{\,j_\ell}=0 \quad (\bmod\ q).
$$
Collect these into the **interpolation matrix**
$$
A\in\mathbb F_q^{\,n\times N},\qquad
A[r,\ell] \;=\; \alpha_r^{\,i_\ell}\,y_r^{\,j_\ell},
$$
and the coefficient vector $c=(c_{i_\ell,j_\ell})_{\ell=1}^N$. Then the interpolation step is exactly the homogeneous system
$$
A\,c \;=\; 0 \quad(\bmod\ q).
$$
Choosing $D$ large enough guarantees $|\mathcal M_D|=N>n$, so the nullspace is non-trivial and a nonzero solution exists.

---

## What `rref_mod` and `nullspace_mod` do

- **`rref_mod(A)`**  
  Performs **Gaussian elimination over $\mathbb F_{257}$** to compute the **Reduced Row Echelon Form** of $A$ and the list of **pivot columns**. All arithmetic is modulo $q=257$ (we use modular inverses of nonzero entries).

- **`nullspace_mod(A)`**  
  Uses the RREF to construct a basis of the **nullspace** $\ker(A)=\{c\in\mathbb F_q^N : A c=0\}$: each non-pivot (free) column yields one nullspace vector by setting that free variable to $1$ and solving the pivot variables from the RREF. Any nonzero vector $c$ from this basis gives coefficients for a valid $Q(X,Y)$.

**In code:**  
1. Build $\mathcal M_D$ and $A$ via `monomials_weighted_deg(...)` and `build_interpolation_matrix(...)`.  
2. Call `nullspace_mod(A)` and take any nonzero vector $c$.  
3. Interpret $c$ on $\mathcal M_D$ to obtain $Q(X,Y)$.

---

## Why this is enough (link to Lemma 12.2.6 and root finding)

For any candidate $P(X)$ with $\deg P\le k-1$, define
$$
R(X)=Q\big(X,\,P(X)\big).
$$
By **Lemma 12.2.6**, $\deg R\le D$. If $P$ **agrees** with the received word on more than $D$ positions, then $R$ has $>D$ distinct zeros, forcing $R\equiv 0$. Equivalently, $Y-P(X)$ is a factor of $Q(X,Y)$.

In this notebook we avoid bivariate factoring: after computing $Q$ from the nullspace we  
1) generate/guess $P$ (brute force for tiny $k$, or sample $k$-tuples),  
2) require **agreement $\ge D+1$**, and  
3) verify that $Q(X,P(X))$ evaluates to zero at **$>D$** of the $\alpha_r$’s (the “lemma check”).

This realizes the **Interpolation + Root-Finding** structure of **Algorithm 12.2.2** using elementary linear algebra over $\mathbb F_{257}$.

In [17]:
def to_int(a: int) -> int:
    a %= q
    return a if a <= q//2 else a - q

def poly_to_str(P, var="x", signed=True):
    """Format coeffs low→high as a polynomial string."""
    terms = []
    for d, c in enumerate(P):
        c = to_int(c) if signed else (c % q)
        if c == 0:
            continue
        if d == 0:
            terms.append(f"{c}")
        elif d == 1:
            terms.append(f"{c}·{var}")
        else:
            terms.append(f"{c}·{var}^{d}")
    return " + ".join(terms) if terms else "0"

def print_results_block(title, n, k, D, t, results, r=None, P_true=None):
    print("="*62)
    print(title)
    print(f"n={n}, k={k} | weighted-degree D={D}, threshold t=D+1={t}")
    if P_true is not None:
        print(f"True P(X): {poly_to_str(P_true)}")
    print("-"*62)
    if not results:
        print("No candidates passed the threshold.")
    else:
        for i, item in enumerate(results, 1):
            P = item["P"]
            line = f"{i:>2}. P(X) = {poly_to_str(P)}"
            line += f" | agreements={item['agreement']}"
            if item.get("zeros_QXP") is not None:
                line += f", zeros(Q(X,P(X)))={item['zeros_QXP']}"
            if r is not None and "codeword" in item:
                # also show disagreements vs received word
                disag = sum(int(a != b) for a, b in zip(item["codeword"], r))
                line += f", disagreements={disag}"
            print(line)
    print("="*62)

def test_two_lines(verbose=True):
    # n=14, xs = {-7,...,-1,1,...,7}, 6 on y=x, 6 on y=-x, 2 noise
    xs = list(range(-7,0)) + list(range(1,8))
    xs = [x % q for x in xs]
    n, k = len(xs), 2
    pos_idx = [3,4,7,9,11,13]
    neg_idx = [0,1,6,8,10,12]
    noise_idx = [2,5]
    ys = [0]*n
    for i in pos_idx: ys[i] = xs[i]               # y = +x
    for i in neg_idx: ys[i] = (-xs[i]) % q        # y = -x
    ys[noise_idx[0]] = 123
    ys[noise_idx[1]] = 77

    D, t = Sudan2DecoderRS.degree_params(n, k)
    results = Sudan2DecoderRS.list_decode(xs, ys, k, D)

    out = {
        "D": D, "t": t, "count": len(results),
        "polys": [tuple(item["P"]) for item in results]
    }
    if verbose:
        print_results_block(
            "Two hidden lines (y=±x) with noise",
            n, k, D, t, results, r=ys
        )
    return out

def test_random_line(n=40, k=2, errors=20, seed=42, verbose=True):
    rng = np.random.default_rng(seed)
    xs = [i+1 for i in range(n)]
    xs = [x % q for x in xs]
    # random line
    a1 = int(rng.integers(0,q)); a0 = int(rng.integers(0,q))
    P_true = [a0, a1]
    c = [eval_poly_univariate(P_true, x) for x in xs]
    ys = c.copy()
    for i in rng.choice(n, size=errors, replace=False):
        ys[i] = int(rng.integers(0, q))

    D, t = Sudan2DecoderRS.degree_params(n, k)
    results = Sudan2DecoderRS.list_decode(xs, ys, k, D)

    out = {
        "D": D, "t": t, "errors": errors,
        "P_true": tuple(P_true),
        "found": any(tuple(item["P"]) == tuple(P_true) for item in results),
        "num_candidates": len(results)
    }
    if verbose:
        print_results_block(
            f"Random line with {errors} errors",
            n, k, D, t, results, r=ys, P_true=P_true
        )
    return out

def test_no_solution(n=20, k=2, seed=7, verbose=True):
    rng = np.random.default_rng(seed)
    xs = [i+1 for i in range(n)]
    xs = [x % q for x in xs]
    ys = [int(rng.integers(0,q)) for _ in range(n)]

    D, t = Sudan2DecoderRS.degree_params(n, k)
    results = Sudan2DecoderRS.list_decode(xs, ys, k, D)

    out = {"D": D, "t": t, "num_candidates": len(results)}
    if verbose:
        print_results_block(
            "Pure noise (expect no candidates)",
            n, k, D, t, results, r=ys
        )
    return out

_ = test_two_lines(verbose=True)
_ = test_random_line(verbose=True)
_ = test_no_solution(verbose=True)

Two hidden lines (y=±x) with noise
n=14, k=2 | weighted-degree D=5, threshold t=D+1=6
--------------------------------------------------------------
 1. P(X) = 1·x | agreements=6, zeros(Q(X,P(X)))=14, disagreements=8
 2. P(X) = -1·x | agreements=6, zeros(Q(X,P(X)))=14, disagreements=8
Random line with 20 errors
n=40, k=2 | weighted-degree D=8, threshold t=D+1=9
True P(X): -59 + 22·x
--------------------------------------------------------------
 1. P(X) = -59 + 22·x | agreements=20, zeros(Q(X,P(X)))=40, disagreements=20
Pure noise (expect no candidates)
n=20, k=2 | weighted-degree D=6, threshold t=D+1=7
--------------------------------------------------------------
No candidates passed the threshold.


In [15]:
# ============================================================
# Demo helpers + a fast (sampling) root-finder for k >= 3
# ============================================================
import numpy as np
from typing import List, Tuple

# --- message <-> polynomial (deg <= k-1) --------------------
def message_to_poly(msg: str, k: int) -> List[int]:
    if len(msg) > k:
        raise ValueError("Message length must be <= k for this simple demo.")
    return [ord(ch) % q for ch in msg] + [0] * (k - len(msg))

def poly_to_message(P: List[int], L: int) -> str:
    bs = [int(c) % 256 for c in P[:L]]
    try:
        return bytes(bs).decode("latin1")
    except Exception:
        return "".join(chr(b) for b in bs)

def hamming_distance(a: List[int], b: List[int]) -> int:
    return sum(int(x != y) for x, y in zip(a, b))

# --- modular solve for "interpolate P through k points" -----
def _solve_poly_through_points(xs_sel: List[int], ys_sel: List[int], k: int) -> List[int] | None:
    """Solve for coeffs P (deg <= k-1) s.t. P(xs_i) = ys_i (i=1..k), mod q."""
    A = np.zeros((k, k), dtype=np.int64)
    b = np.array([y % q for y in ys_sel], dtype=np.int64)
    for r in range(k):
        A[r, 0] = 1
        for c in range(1, k):
            A[r, c] = (A[r, c-1] * (xs_sel[r] % q)) % q
    Aug = np.concatenate([A % q, b.reshape(-1, 1)], axis=1) % q
    R, piv = rref_mod(Aug)
    if len(piv) < k:
        return None
    return [int(R[i, -1] % q) for i in range(k)]

# --- sampled root-finder: try random k-tuples, validate by Q ---
def list_decode_sampled(xs: List[int],
                        ys: List[int],
                        k: int,
                        D: int,
                        verify: bool = True,
                        return_codewords: bool = True,
                        samples: int = 6000,
                        seed: int = 0):
    rng = np.random.default_rng(seed)
    xs = [x % q for x in xs]; ys = [y % q for y in ys]

    data = Sudan2DecoderRS.interpolate_Q(xs, ys, k, D)
    D, t, mons, cvec = data["D"], data["t"], data["mons"], data["cvec"]

    out, seen = [], set()
    n = len(xs)
    for _ in range(samples):
        idx = rng.choice(n, size=k, replace=False)
        P = _solve_poly_through_points([xs[i] for i in idx], [ys[i] for i in idx], k)
        if P is None: 
            continue
        key = tuple(P)
        if key in seen:
            continue
        seen.add(key)

        agree = sum(int(eval_poly_univariate(P, x) == y) for x, y in zip(xs, ys))
        if agree < (D + 1):
            continue

        zeros = None
        if verify:
            zeros = sum(int(Sudan2DecoderRS._eval_Q(cvec, mons, x, eval_poly_univariate(P, x)) == 0)
                        for x in xs)
            if zeros <= D:
                continue

        item = {"P": P, "agreement": agree, "zeros_QXP": zeros}
        if return_codewords:
            item["codeword"] = [eval_poly_univariate(P, x) for x in xs]
        out.append(item)

    out.sort(key=lambda d: (-d["agreement"], d["P"]))
    return out

# --- Part-1 style demo printout for Sudan-2 + thresholds -----
def RS_demonstration(msg: str, n: int, k: int, err: int, seed: int = 0, xs=None,
                     samples: int = 8000, use_sampling: bool = True):
    """
    Prints a readable run:
      * shows unique-decoding radius
      * shows Part-1 (Alg1) agreement threshold t > 2*sqrt(n*(k-1))
      * shows Sudan-2 (Alg2) agreement threshold t > sqrt(2*(k-1)*n)
      * runs Sudan-2; for k>=3 defaults to 'sampled' root finding for speed
    """
    rng = np.random.default_rng(seed)
    if xs is None:
        xs = [i + 1 for i in range(n)]
    xs = [x % q for x in xs]

    P_true = message_to_poly(msg, k)
    c = [eval_poly_univariate(P_true, x) for x in xs]

    r = c.copy()
    for i in rng.choice(n, size=err, replace=False):
        r[i] = int(rng.integers(0, q))

    D2, _ = Sudan2DecoderRS.degree_params(n, k)  # Alg2 choice
    t_actual = n - err
    unique_rad = (n - k) // 2
    t1_req = 2.0 * (n * (k - 1)) ** 0.5                # Part-1 (Alg1) threshold
    t2_req = (2.0 * (k - 1) * n) ** 0.5                # Sudan-2 threshold

    print("=" * 62)
    print(f"--- Running Test: msg='{msg}', n={n}, k={k}, errors={err} ---")
    print(f"Unique decoding can handle at most {unique_rad} errors.")
    if err > unique_rad:
        print("NOTE: Number of errors exceeds unique-decoding limit. List decoding is required.")
    print("-" * 62)

    print("\nOriginal Codeword (first 30 symbols):")
    print(c[:30])
    print("\nNoisy Codeword (first 30 symbols):")
    print(r[:30])

    print("\n--- Starting List Decoding (Sudan Algorithm 2) ---")
    info = Sudan2DecoderRS.interpolate_Q(xs, r, k, D2)
    D2 = info["D"]; t2 = D2 + 1
    print(f"Interpolation: solving nullspace of a {n}x{len(info['mons'])} matrix;  D={D2}, threshold t={t2}.")

    if use_sampling and k >= 3:
        results = list_decode_sampled(xs, r, k, D2, verify=True, return_codewords=True,
                                      samples=samples, seed=seed + 1)
    else:
        # Warning: brute-force explodes for k>=3
        results = Sudan2DecoderRS.list_decode(xs, r, k, D2, verify=True, return_codewords=True)

    print(f"\nCandidates found: {len(results)}")
    if results:
        dists = [hamming_distance(item["codeword"], r) for item in results]
        mind = min(dists)
        print("\n--- Filtering Candidates by Agreement ---")
        for item, d in zip(results, dists):
            print(f"  - Candidate P(X)={item['P']} has {n - d} agreements ({d} disagreements).")

        print(f"\nMinimum distance among candidates is {mind}.")
        keep = [item for item, d in zip(results, dists) if d == mind]
        print("Accepting all candidates with minimum distance:")
        print([item["P"] for item in keep])

        msgs = [poly_to_message(item["P"], len(msg)) for item in keep]
    else:
        msgs = []

    print("\n--- Final Results ---")
    print("Output Polynomials List:")
    print([item["P"] for item in results])
    print("\nDecoded Possible Messages:")
    print(msgs)

    success = any(tuple(item["P"]) == tuple(P_true) for item in results)

    print("\nComparison vs thresholds:")
    print(f"  agreements t = n - err = {t_actual}")
    print(f"  Unique decoding radius  ⩾ {(n-k)//2} errors  -> {'PASS' if err <= unique_rad else 'FAIL'}")
    print(f"  Alg1 needs t > 2·√(n(k-1)) ≈ {t1_req:.2f}     -> {'PASS' if t_actual > t1_req else 'FAIL'}")
    print(f"  Alg2 needs t > √(2(k-1)n) ≈ {t2_req:.2f}      -> {'PASS' if t_actual > t2_req else 'FAIL'}")

    print("\n" + ("SUCCESS: Original message was recovered in the list!"
                   if success else "FAILURE: Could not recover original message."))
    print("=" * 62)

    return {"success": success, "results": results, "P_true": P_true, "xs": xs, "r": r}


# Example runs (similar process as Part 1)

# Test 1: should succeed (n=40, k=5, err=10 -> t=30)
_ = RS_demonstration(msg="abcde", n=40, k=5, err=10, seed=1, samples=8000, use_sampling=True)

# Test 2: a failing case (n=20, k=5, err=8 -> t=12, below both Alg1 and Alg2 thresholds)
_ = RS_demonstration(msg="hello", n=20, k=5, err=8, seed=2, samples=8000, use_sampling=True)

--- Running Test: msg='abcde', n=40, k=5, errors=10 ---
Unique decoding can handle at most 17 errors.
--------------------------------------------------------------

Original Codeword (first 30 symbols):
[238, 21, 84, 148, 45, 232, 249, 4, 30, 200, 241, 248, 170, 67, 110, 67, 74, 121, 52, 79, 11, 25, 152, 20, 139, 102, 127, 29, 248, 50]

Noisy Codeword (first 30 symbols):
[238, 215, 84, 148, 45, 84, 249, 4, 30, 31, 241, 248, 7, 67, 222, 67, 210, 121, 52, 79, 11, 25, 152, 20, 116, 102, 127, 29, 248, 50]

--- Starting List Decoding (Sudan Algorithm 2) ---
Interpolation: solving nullspace of a 40x50 matrix;  D=17, threshold t=18.

Candidates found: 1

--- Filtering Candidates by Agreement ---
  - Candidate P(X)=[97, 98, 99, 100, 101] has 30 agreements (10 disagreements).

Minimum distance among candidates is 10.
Accepting all candidates with minimum distance:
[[97, 98, 99, 100, 101]]

--- Final Results ---
Output Polynomials List:
[[97, 98, 99, 100, 101]]

Decoded Possible Messages:
['abc